# Member 4 — Dataset Exploration and Analysis

**Group 6 · Image Captioning / Sparse-BM25 RAG · Phase 1**

This notebook performs the statistical and visual exploration assigned to Member 4. It reads the repository files directly and applies the v1.1 caption exclusion list before calculating final KB caption statistics.

> **Important data-version rule:** the repository snapshot contains `data/captions_processed.json` with the older 55,500-caption total. The v1.1 handoff specifies 10,000 KB images and 49,777 KB captions after excluding 223 captions. Therefore, this notebook uses `splits/kb_manifest.json` as the raw KB source and joins cleaned captions from `data/captions_processed.json`, excluding records by the exact `(kb_image_id, raw caption)` pair. It never treats the old 55,500 total as the final KB total.

**What this notebook can calculate from the ZIP:** object statistics, caption statistics, vocabulary, optional POS tags, object co-occurrence, and approximate spatial relationships from bounding boxes. **What it cannot establish from this ZIP alone:** image colour, indoor/outdoor, and day/night distributions, because the ZIP does not contain the 10,000 image files or scene labels. Those sections are explicitly marked as pending rather than guessed.


## 1. Environment and imports

Run this notebook from the repository root, or set `REPO_ROOT` manually in the next cell. In Google Colab, upload/extract the repository ZIP first. Required core libraries: `pandas`, `numpy`, and `matplotlib`.


In [ ]:
from pathlib import Path
import json, re, itertools, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# If the notebook is inside repository/exploration/, the parent is the repository root.
# If running in Colab after extracting the ZIP, update REPO_ROOT if necessary.
CANDIDATES = [
    Path('/content/project_extracted/image_processing_project-main'),
    Path('/content/image_processing_project-main'),
    Path.cwd(),
    Path.cwd().parent,
]
REPO_ROOT = next((p for p in CANDIDATES if (p/'data/metadata.json').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Repository root not found. Set REPO_ROOT = Path("/path/to/image_processing_project-main") manually.')

DATA_DIR = REPO_ROOT / 'data'
SPLITS_DIR = REPO_ROOT / 'splits'
EXPLORATION_DIR = REPO_ROOT / 'exploration'
VIS_DIR = EXPLORATION_DIR / 'visualizations'
EXPLORATION_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = VIS_DIR  # Existing plot cells use OUT_DIR; tables are exported to EXPLORATION_DIR.
print('Repository:', REPO_ROOT)
print('Visualizations will be saved to:', OUT_DIR)


## 2. Load the actual repository files

The metadata records contain `image_id`, `split`, and detected objects (`label`, `conf`, `bbox`). The KB manifest contains the five raw captions for each KB image. The exclusion list is expected by the handoff to be `splits/excluded_captions.json`; the uploaded ZIP has the same file named `excluded_captions (1).json`, so the loader accepts either filename without changing the repository.


In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

metadata_obj = load_json(DATA_DIR / 'metadata.json')
metadata = metadata_obj['data']
caption_candidates = [
    DATA_DIR / 'captions_processed_v1.1.json',
    DATA_DIR / 'captions_processed.json',
]
caption_path = next((p for p in caption_candidates if p.exists()), None)
if caption_path is None:
    raise FileNotFoundError('Neither captions_processed_v1.1.json nor captions_processed.json was found.')
captions_processed = load_json(caption_path)
print('Caption source:', caption_path.name)
kb_manifest = load_json(SPLITS_DIR / 'kb_manifest.json')

exclusion_candidates = [SPLITS_DIR/'excluded_captions.json', SPLITS_DIR/'excluded_captions (1).json']
exclusion_path = next((p for p in exclusion_candidates if p.exists()), None)
if exclusion_path is None:
    raise FileNotFoundError('Could not find excluded_captions.json (or the ZIP variant with (1)).')
excluded = load_json(exclusion_path)

print('Detector:', metadata_obj.get('detector'))
print('Ultralytics version:', metadata_obj.get('ultralytics_version'))
print('Confidence threshold:', metadata_obj.get('conf_threshold'))
print('Metadata records:', len(metadata))
print('Processed caption records:', len(captions_processed))
print('KB manifest images:', len(kb_manifest))
print('Excluded caption entries:', len(excluded))
print('Exclusion file:', exclusion_path.name)


## 3. Verify v1.1 data integrity before analysis

The handoff states that the KB has 10,000 images, 50,000 original captions, and 49,777 captions after excluding 223 captions. It also says no image is left without captions. These are validation targets, not values to force: if an assertion fails, stop and investigate the source files before reporting results.


In [ ]:
from collections import defaultdict

# Use exact raw caption strings as required by the handoff.
excluded_pairs = {(int(e['kb_image_id']), e['caption']) for e in excluded}
raw_kb_caption_count = sum(len(item['captions']) for item in kb_manifest)
excluded_pair_count = sum((int(item['image_id']), cap) in excluded_pairs
                          for item in kb_manifest for cap in item['captions'])
remaining_kb_caption_count = raw_kb_caption_count - excluded_pair_count
images_with_zero_remaining = []
for item in kb_manifest:
    kept = [c for c in item['captions'] if (int(item['image_id']), c) not in excluded_pairs]
    if not kept:
        images_with_zero_remaining.append(item['image_id'])

validation = pd.DataFrame([
    ['KB images', len(kb_manifest), 10000],
    ['Raw KB captions', raw_kb_caption_count, 50000],
    ['Matched exclusions applied', excluded_pair_count, 223],
    ['KB captions after exclusion', remaining_kb_caption_count, 49777],
    ['Images with zero captions after exclusion', len(images_with_zero_remaining), 0],
], columns=['Check', 'Observed', 'Expected from v1.1 handoff'])
display(validation)

assert len(kb_manifest) == 10000, 'KB image count does not match v1.1.'
assert raw_kb_caption_count == 50000, 'Raw KB caption count does not match expected 50,000.'
assert excluded_pair_count == 223, 'Not all 223 exclusions matched the raw KB manifest.'
assert remaining_kb_caption_count == 49777, 'Post-exclusion KB caption count is not 49,777.'
assert len(images_with_zero_remaining) == 0, 'At least one KB image has no captions after exclusion.'
print('PASS: v1.1 KB integrity checks passed.')


## 4. Build the final KB caption table (raw + cleaned)

The processed-caption JSON is joined to the manifest by image ID. Raw and cleaned captions are paired by their original list position. The exclusion decision is made using the raw caption string, exactly as required by the team handoff. `captions_raw` is not modified; only retrieval/analysis rows are filtered.


In [ ]:
processed_by_id = {int(row['image_id']): row for row in captions_processed}
caption_rows = []
missing_processed_ids = []
for item in kb_manifest:
    image_id = int(item['image_id'])
    proc = processed_by_id.get(image_id)
    if proc is None:
        missing_processed_ids.append(image_id)
        continue
    raw_list = item['captions']
    proc_raw = proc.get('captions_raw', [])
    clean_list = proc.get('captions_clean', [])
    if len(raw_list) != len(clean_list):
        raise ValueError(f'Raw/clean caption list length mismatch for image {image_id}')
    # Align processed cleaned text to the manifest's raw captions by exact raw string.
    clean_by_raw = {}
    for raw, clean in zip(proc_raw, clean_list):
        clean_by_raw.setdefault(raw, clean)
    for raw in raw_list:
        if (image_id, raw) in excluded_pairs:
            continue
        clean = clean_by_raw.get(raw)
        if clean is None:
            raise ValueError(f'Cleaned caption not found for image {image_id}: {raw[:80]}')
        caption_rows.append({'image_id': image_id, 'caption_raw': raw, 'caption_clean': clean})

if missing_processed_ids:
    raise ValueError(f'Missing processed-caption records for {len(missing_processed_ids)} KB images.')

captions_df = pd.DataFrame(caption_rows)
print('Final KB caption rows:', len(captions_df))
print('Unique KB image IDs in caption table:', captions_df.image_id.nunique())
display(captions_df.head())
assert len(captions_df) == 49777
assert captions_df.image_id.nunique() == 10000


## 5. Before/after preprocessing comparison

For the KB, preprocessing exclusions are measured against the original manifest: 50,000 raw captions before leakage filtering, 223 excluded captions, and 49,777 retained captions. This is the caption-level exclusion described in v1.1; it is not the same as lowercasing or punctuation cleaning.


In [ ]:
before_after = pd.DataFrame({
    'Stage': ['Before v1.1 leakage exclusion', 'Excluded by v1.1 list', 'After v1.1 leakage exclusion'],
    'Caption count': [raw_kb_caption_count, excluded_pair_count, remaining_kb_caption_count]
})
display(before_after)
plt.figure(figsize=(8, 4))
plt.bar(before_after['Stage'], before_after['Caption count'])
plt.title('KB captions before and after v1.1 exclusion')
plt.ylabel('Number of captions')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(OUT_DIR/'caption_exclusion_comparison.png', dpi=160)
plt.show()


## 6. Object statistics

Object counts are computed from `data/metadata.json`, restricted to `split == "kb"`. Each object instance contributes one count. The average objects per image includes KB images with zero detections, if any.


In [ ]:
kb_metadata = [row for row in metadata if row.get('split') == 'kb']
assert len(kb_metadata) == 10000, f'Expected 10,000 KB metadata records; found {len(kb_metadata)}'
object_rows = []
for row in kb_metadata:
    image_id = int(row['image_id'])
    for obj in row.get('objects', []):
        object_rows.append({'image_id': image_id, 'label': str(obj['label']).lower(),
                            'confidence': float(obj.get('conf', np.nan)),
                            'bbox': obj.get('bbox', [])})
objects_df = pd.DataFrame(object_rows)
objects_per_image = pd.Series({int(r['image_id']): len(r.get('objects', [])) for r in kb_metadata}, name='object_count')
object_counts = objects_df['label'].value_counts()
object_summary = pd.Series({
    'KB images': len(kb_metadata),
    'Total detected object instances': len(objects_df),
    'Unique object classes': objects_df['label'].nunique(),
    'Average objects per image': objects_per_image.mean(),
    'Minimum objects per image': objects_per_image.min(),
    'Maximum objects per image': objects_per_image.max(),
    'Images with no detected objects': int((objects_per_image == 0).sum())
})
display(object_summary.to_frame('Value'))
display(object_counts.head(50).rename('frequency').to_frame())


In [ ]:
top50 = object_counts.head(50).sort_values()
plt.figure(figsize=(10, 13))
top50.plot(kind='barh')
plt.title('Top 50 detected object classes in the KB')
plt.xlabel('Number of detected instances')
plt.ylabel('Object class')
plt.tight_layout()
plt.savefig(OUT_DIR/'top_50_object_classes.png', dpi=160)
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(objects_per_image, bins=30, edgecolor='black')
plt.title('Distribution of detected objects per KB image')
plt.xlabel('Number of detected objects')
plt.ylabel('Number of images')
plt.tight_layout()
plt.savefig(OUT_DIR/'objects_per_image_histogram.png', dpi=160)
plt.show()


## 7. Caption length, vocabulary, and frequent words

Caption length is measured in whitespace-separated words after the repository's cleaning step. Vocabulary is the set of unique cleaned tokens. These statistics use the 49,777 retained KB captions only.


In [ ]:
def words(text):
    return str(text).split()

captions_df['word_count'] = captions_df['caption_clean'].map(lambda s: len(words(s)))
all_tokens = [tok for text in captions_df['caption_clean'] for tok in words(text)]
token_counts = Counter(all_tokens)
caption_summary = pd.Series({
    'Retained KB captions': len(captions_df),
    'Average caption length (words)': captions_df.word_count.mean(),
    'Minimum caption length (words)': captions_df.word_count.min(),
    'Maximum caption length (words)': captions_df.word_count.max(),
    'Vocabulary size': len(token_counts),
    'Excluded KB captions': excluded_pair_count
})
display(caption_summary.to_frame('Value'))
display(pd.DataFrame(token_counts.most_common(30), columns=['token','frequency']))

plt.figure(figsize=(8, 4))
plt.hist(captions_df.word_count, bins=30, edgecolor='black')
plt.title('Cleaned KB caption length distribution')
plt.xlabel('Words per caption')
plt.ylabel('Number of captions')
plt.tight_layout()
plt.savefig(OUT_DIR/'caption_length_histogram.png', dpi=160)
plt.show()


## 8. Part-of-speech (POS) distribution

POS tagging requires a language tagger. This cell uses NLTK's English averaged-perceptron tagger. If the tagger data is not available in the environment, install/download it and rerun this cell. POS results are model-generated linguistic estimates, not manually verified labels.


In [ ]:
try:
    import nltk
    try:
        nltk.data.find('taggers/averaged_perceptron_tagger_eng')
    except LookupError:
        nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    tagged_words = []
    for caption in captions_df['caption_clean']:
        tagged_words.extend(nltk.pos_tag(words(caption)))
    pos_counts = Counter(tag for _, tag in tagged_words)
    pos_groups = Counter()
    for tag, count in pos_counts.items():
        if tag.startswith('NN'): group = 'Nouns'
        elif tag.startswith('VB'): group = 'Verbs'
        elif tag.startswith('JJ'): group = 'Adjectives'
        elif tag.startswith('RB'): group = 'Adverbs'
        else: group = 'Other'
        pos_groups[group] += count
    pos_df = pd.DataFrame(pos_groups.items(), columns=['POS category','Token count']).sort_values('Token count', ascending=False)
    display(pos_df)
    plt.figure(figsize=(7, 4))
    plt.bar(pos_df['POS category'], pos_df['Token count'])
    plt.title('Part-of-speech distribution in cleaned KB captions')
    plt.ylabel('Tagged token count')
    plt.tight_layout()
    plt.savefig(OUT_DIR/'pos_distribution.png', dpi=160)
    plt.show()
except Exception as exc:
    print('POS analysis was not completed in this environment:', repr(exc))
    print('Install NLTK, download averaged_perceptron_tagger_eng, then rerun this section.')


## 9. Object co-occurrence

Co-occurrence counts how many KB images contain each pair of object classes at least once. It is based on detector labels, not caption co-mentions. To keep the heatmap readable, the default uses the 20 most frequent classes.


In [ ]:
from collections import defaultdict
image_labels = {}
for row in kb_metadata:
    image_labels[int(row['image_id'])] = sorted(set(str(o['label']).lower() for o in row.get('objects', [])))
co_classes = list(object_counts.head(20).index)
co_matrix = pd.DataFrame(0, index=co_classes, columns=co_classes, dtype=int)
for labels in image_labels.values():
    present = [x for x in labels if x in co_matrix.index]
    for a in present:
        for b in present:
            co_matrix.loc[a,b] += 1

plt.figure(figsize=(12, 10))
plt.imshow(co_matrix.values, aspect='auto')
plt.colorbar(label='Number of images containing both classes')
plt.xticks(range(len(co_classes)), co_classes, rotation=90)
plt.yticks(range(len(co_classes)), co_classes)
plt.title('Object-class co-occurrence (top 20 classes)')
plt.tight_layout()
plt.savefig(OUT_DIR/'object_cooccurrence_top20.png', dpi=160)
plt.show()


## 10. Spatial relationships from bounding boxes

The metadata stores boxes as `[x1, y1, x2, y2]`, but does not store the original image width and height in each record. We can still compare object centers within the same image to estimate left/right and above/below relationships. For the scatter plot, centers are scaled by the maximum box coordinates observed in that image; this is an **approximation**, not exact image-coordinate normalization.

The notebook reports directional pair counts and box-area ratios. It does not claim semantic relations such as “holding” or “riding”.


In [ ]:
spatial_pairs = Counter()
center_rows = []
area_rows = []
for row in kb_metadata:
    objs = row.get('objects', [])
    valid = []
    for obj in objs:
        bb = obj.get('bbox', [])
        if len(bb) != 4: continue
        x1,y1,x2,y2 = map(float, bb)
        if x2 <= x1 or y2 <= y1: continue
        cx, cy = (x1+x2)/2, (y1+y2)/2
        valid.append((str(obj['label']).lower(), cx, cy, x1,y1,x2,y2))
        area_rows.append({'label': str(obj['label']).lower(), 'box_area': (x2-x1)*(y2-y1)})
    if valid:
        max_x = max(v[5] for v in valid) or 1.0
        max_y = max(v[6] for v in valid) or 1.0
        for label,cx,cy,*_ in valid:
            center_rows.append({'image_id': int(row['image_id']), 'label':label, 'center_x_approx':cx/max_x, 'center_y_approx':cy/max_y})
    for a,b in itertools.combinations(valid,2):
        la,xa,ya,*_ = a; lb,xb,yb,*_ = b
        if abs(xa-xb) >= abs(ya-yb):
            rel = 'left-right'
            key = (la,lb) if xa < xb else (lb,la)
        else:
            rel = 'above-below'
            key = (la,lb) if ya < yb else (lb,la)
        spatial_pairs[(rel,key[0],key[1])] += 1

centers_df = pd.DataFrame(center_rows)
areas_df = pd.DataFrame(area_rows)
spatial_df = pd.DataFrame([{'relationship':k[0], 'first_class':k[1], 'second_class':k[2], 'pair_count':v}
                           for k,v in spatial_pairs.most_common(30)])
display(spatial_df.head(30))
display(areas_df.groupby('label').box_area.median().sort_values(ascending=False).head(20).to_frame('median_bbox_area_pixels_squared'))


In [ ]:
if not centers_df.empty:
    plot_labels = set(object_counts.head(8).index)
    subset = centers_df[centers_df.label.isin(plot_labels)]
    plt.figure(figsize=(8, 6))
    for label, group in subset.groupby('label'):
        plt.scatter(group.center_x_approx, group.center_y_approx, s=8, alpha=0.25, label=label)
    plt.gca().invert_yaxis()
    plt.xlabel('Approximate normalized object-center X')
    plt.ylabel('Approximate normalized object-center Y')
    plt.title('Approximate spatial distribution of frequent object classes')
    plt.legend(bbox_to_anchor=(1.02,1), loc='upper left')
    plt.tight_layout()
    plt.savefig(OUT_DIR/'object_center_scatter.png', dpi=160)
    plt.show()


## 11. Bounding-box shape and size analysis

The repository has bounding boxes but no explicit object-shape annotations. Width/height ratio can describe the geometry of detected boxes (wide, tall, or approximately square); it must not be interpreted as the true physical shape of an object.


In [ ]:
bbox_records = []
for row in kb_metadata:
    for obj in row.get('objects', []):
        bb = obj.get('bbox', [])
        if len(bb) == 4:
            x1,y1,x2,y2 = map(float,bb)
            w,h = x2-x1,y2-y1
            if w > 0 and h > 0:
                bbox_records.append({'label':str(obj['label']).lower(),'width':w,'height':h,'aspect_ratio':w/h,'area':w*h})
bbox_df = pd.DataFrame(bbox_records)
bbox_df['box_shape_group'] = np.select([bbox_df.aspect_ratio > 1.2, bbox_df.aspect_ratio < 1/1.2], ['Wide box','Tall box'], default='Approximately square box')
display(bbox_df.box_shape_group.value_counts().to_frame('count'))
plt.figure(figsize=(7,4))
bbox_df.box_shape_group.value_counts().reindex(['Wide box','Approximately square box','Tall box']).plot(kind='bar')
plt.title('Bounding-box geometry (proxy for shape)')
plt.ylabel('Detected object instances')
plt.tight_layout()
plt.savefig(OUT_DIR/'bbox_shape_groups.png', dpi=160)
plt.show()


## 12. Image-level diversity (requires the correct image files)

**Not computable from the uploaded repository ZIP:** it contains metadata and captions, but no `kb_images/` directory. The progress report also records a previous image/metadata ID mismatch, so do not use an unverified image folder.

If the team supplies the v1.1 KB image directory, set `KB_IMAGE_DIR` below. The code checks exact filenames against `kb_manifest.json` before calculating dominant average RGB colour. A colour estimate is not an indoor/outdoor or day/night classifier. Those scene categories require a documented classifier or manually verified labels.


In [ ]:
# Set this to the verified 10,000-image Google Drive folder.
# This path was used for the user's original KB images in Colab.
KB_IMAGE_DIR = Path('/content/drive/MyDrive/G6_RAG_Dataset/kb_images')

from PIL import Image
from collections import Counter
import colorsys

def classify_color(r, g, b):
    h, sat, val = colorsys.rgb_to_hsv(r/255, g/255, b/255)
    h *= 360
    sat *= 100
    val *= 100
    if val < 18: return "Black"
    if sat < 18: return "White" if val > 80 else "Gray"
    if 10 <= h < 35 and val < 65: return "Brown"
    if h < 10 or h >= 345: return "Red"
    if h < 25: return "Orange"
    if h < 45: return "Yellow"
    if h < 85: return "Green"
    if h < 100: return "Cyan"
    if h < 135: return "Blue"
    if h < 165: return "Purple"
    return "Pink"

if KB_IMAGE_DIR.exists():
    pixel_counts, dominant_image_counts = Counter(), Counter()
    image_files = sorted(KB_IMAGE_DIR.glob("*.jpg"))
    print("JPG images found:", len(image_files))
    for idx, image_path in enumerate(image_files, 1):
        with Image.open(image_path) as im:
            pixels = np.asarray(im.convert("RGB").resize((32, 32))).reshape(-1, 3)
        per_image = Counter(classify_color(int(r), int(g), int(b)) for r,g,b in pixels)
        pixel_counts.update(per_image)
        if per_image:
            dominant_image_counts[per_image.most_common(1)[0][0]] += 1
        if idx % 1000 == 0:
            print(f"Processed {idx} images")
    color_df = pd.DataFrame({
        "Color": sorted(pixel_counts),
        "Sampled pixel count": [pixel_counts[c] for c in sorted(pixel_counts)],
        "Images where dominant": [dominant_image_counts[c] for c in sorted(pixel_counts)]
    })
    color_df.to_csv(EXPLORATION_DIR / "06_color_statistics.csv", index=False)
    plt.figure(figsize=(9,5))
    plt.bar(color_df["Color"], color_df["Sampled pixel count"])
    plt.title("Approximate color distribution across KB images")
    plt.ylabel("Sampled pixels (32×32 per image)")
    plt.tight_layout()
    plt.savefig(VIS_DIR / "fig09_color_distribution.png", dpi=180)
    plt.show()

    dominant_df = color_df.sort_values("Images where dominant", ascending=False)
    plt.figure(figsize=(9,5))
    plt.bar(dominant_df["Color"], dominant_df["Images where dominant"])
    plt.title("Dominant color category per KB image")
    plt.ylabel("Number of images")
    plt.tight_layout()
    plt.savefig(VIS_DIR / "fig10_dominant_colors.png", dpi=180)
    plt.show()
else:
    print("Color analysis skipped: image folder not found:", KB_IMAGE_DIR)

print("Indoor/outdoor and day/night labels are not available in the supplied metadata.")

## 13. Export the computed tables

Save the main numeric outputs so they can be reused in `docs/section_4_4_exploration_and_analysis.md`.


In [ ]:
object_counts.rename_axis('object_class').reset_index(name='frequency').to_csv(OUT_DIR/'object_class_frequencies.csv', index=False)
pd.DataFrame({'image_id':objects_per_image.index,'object_count':objects_per_image.values}).to_csv(OUT_DIR/'objects_per_image.csv', index=False)
captions_df.to_csv(OUT_DIR/'kb_captions_after_v1_1_exclusion.csv', index=False)
co_matrix.to_csv(OUT_DIR/'object_cooccurrence_top20.csv')
spatial_df.to_csv(OUT_DIR/'spatial_relationships_top30.csv', index=False)

summary = {
    'kb_images': int(len(kb_manifest)),
    'raw_kb_captions_before_exclusion': int(raw_kb_caption_count),
    'excluded_kb_captions': int(excluded_pair_count),
    'kb_captions_after_exclusion': int(len(captions_df)),
    'detected_object_instances': int(len(objects_df)),
    'unique_object_classes': int(objects_df.label.nunique()),
    'average_objects_per_image': float(objects_per_image.mean()),
    'average_caption_length_words': float(captions_df.word_count.mean()),
    'vocabulary_size': int(len(token_counts))
}
with open(OUT_DIR/'exploration_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))
print('Saved outputs in:', OUT_DIR)


## 14. Interpretation and limitations (complete after running)

Use the generated tables and figures to write evidence-based observations. Do not copy old preliminary statistics into the final report. In particular:

- State the verified v1.1 KB caption counts (50,000 before exclusion; 223 excluded; 49,777 retained).
- Report object and caption statistics from the executed cells above.
- Explain that object detections are YOLOv8m outputs at the repository's stored confidence threshold; they are not ground-truth annotations.
- Describe spatial relationships as estimates from bounding-box centers.
- Describe box aspect ratios as box geometry, not object shape labels.
- Mark image colour analysis as pending until matching v1.1 images are available.
- Do not claim indoor/outdoor or day/night distributions without verified labels or a validated classifier.

**Before submitting:** run all cells from top to bottom, inspect every assertion, save the notebook with outputs, and verify that all generated figures are present in `exploration/visualizations/`.


## 15. Standardized outputs (tables 01–07, figures fig01–fig10)

This final cell exports the main analysis tables to the `exploration/` folder using numbered filenames. Figures are saved in `exploration/visualizations/`. It does not invent POS or colour results if those analyses were skipped.

In [ ]:
# Export numbered tables. Each table is saved in exploration/, not visualizations/.
table_exports = {}

table_exports["01_validation_checks.csv"] = validation
table_exports["02_caption_before_after.csv"] = before_after
table_exports["03_object_class_frequencies.csv"] = (
    object_counts.rename_axis("object_class").reset_index(name="frequency")
)
table_exports["04_caption_statistics.csv"] = caption_summary.rename_axis("metric").reset_index(name="value")
table_exports["05_object_summary.csv"] = object_summary.rename_axis("metric").reset_index(name="value")
table_exports["06_spatial_relationships.csv"] = spatial_df

# POS table is optional; create a clear note if the tagger did not run.
if "pos_df" in globals():
    table_exports["07_pos_distribution.csv"] = pos_df
else:
    table_exports["07_pos_distribution.csv"] = pd.DataFrame(
        [{"POS category": "Not available", "Token count": None}]
    )

for filename, dataframe in table_exports.items():
    dataframe.to_csv(EXPLORATION_DIR / filename, index=False)

# Additional raw-vs-cleaned comparison, measured only on retained KB caption pairs.
def word_len(value):
    return len(str(value).split())

raw_clean_stats = pd.DataFrame([
    {"Version": "Raw retained KB captions", "Caption count": len(captions_df),
     "Average words": captions_df["caption_raw"].map(word_len).mean(),
     "Median words": captions_df["caption_raw"].map(word_len).median(),
     "Average characters": captions_df["caption_raw"].astype(str).str.len().mean(),
     "Unique captions": captions_df["caption_raw"].nunique()},
    {"Version": "Cleaned retained KB captions", "Caption count": len(captions_df),
     "Average words": captions_df["caption_clean"].map(word_len).mean(),
     "Median words": captions_df["caption_clean"].map(word_len).median(),
     "Average characters": captions_df["caption_clean"].astype(str).str.len().mean(),
     "Unique captions": captions_df["caption_clean"].nunique()},
])
raw_clean_stats.to_csv(EXPLORATION_DIR / "07_raw_vs_cleaned_caption_statistics.csv", index=False)

# Object diversity per image: number of distinct detected classes per KB image.
diversity_rows = []
for image_id, labels in image_labels.items():
    diversity_rows.append({"image_id": image_id, "unique_object_classes": len(labels)})
diversity_df = pd.DataFrame(diversity_rows)
diversity_df.to_csv(EXPLORATION_DIR / "08_object_diversity_per_image.csv", index=False)

summary = {
    "kb_images": int(len(kb_manifest)),
    "raw_kb_captions_before_exclusion": int(raw_kb_caption_count),
    "excluded_kb_captions": int(excluded_pair_count),
    "kb_captions_after_exclusion": int(len(captions_df)),
    "detected_object_instances": int(len(objects_df)),
    "unique_object_classes": int(objects_df["label"].nunique()),
    "average_objects_per_image": float(objects_per_image.mean()),
    "average_caption_length_words": float(captions_df["word_count"].mean()),
    "vocabulary_size": int(len(token_counts)),
    "spatial_relationship_pairs_reported": int(sum(spatial_pairs.values())),
    "object_cooccurrence_classes": int(len(co_classes)),
    "mean_unique_object_classes_per_image": float(diversity_df["unique_object_classes"].mean()),
    "pos_analysis_available": bool("pos_df" in globals()),
    "color_analysis_available": bool((EXPLORATION_DIR / "06_color_statistics.csv").exists()),
}
with open(EXPLORATION_DIR / "exploration_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved numbered tables and exploration_summary.json in:", EXPLORATION_DIR)
print("\nTable files:")
for filename in sorted(table_exports):
    print(" -", filename)
print(" - 07_raw_vs_cleaned_caption_statistics.csv")
print(" - 08_object_diversity_per_image.csv")
print("\nFigure files currently in visualizations:")
for path in sorted(VIS_DIR.glob("*.png")):
    print(" -", path.name)
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Standardize figure names after the original analysis cells have generated their plots.
figure_name_map = {
    "caption_exclusion_comparison.png": "fig01_caption_before_after.png",
    "top_50_object_classes.png": "fig02_top50_object_classes.png",
    "objects_per_image_histogram.png": "fig03_objects_per_image.png",
    "caption_length_histogram.png": "fig04_caption_length.png",
    "pos_distribution.png": "fig05_pos_distribution.png",
    "object_cooccurrence_top20.png": "fig06_object_cooccurrence.png",
    "object_center_scatter.png": "fig07_spatial_centers.png",
    "bbox_shape_groups.png": "fig08_bbox_geometry.png",
}
for old_name, new_name in figure_name_map.items():
    old_path = VIS_DIR / old_name
    if old_path.exists():
        new_path = VIS_DIR / new_name
        new_path.write_bytes(old_path.read_bytes())
print("Standardized figure copies created where source plots exist.")